In [1]:
import sys
sys.path.append('..')

from hybrid_rag.pipeline import HybridRAGPipeline

hybrid = HybridRAGPipeline()

🔧 Initialising Hybrid RAG Pipeline...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ ChromaDB loaded — 7913 child vectors
✅ BM25 loaded — 7913 children
Mistral client ready - model: mistral-medium-latest
✅ Hybrid RAG ready — 7913 vectors | 7913 BM25 children | 2010 parents



In [2]:
from vector_rag.pipeline     import VectorRAGPipeline
from vectorless_rag.pipeline import VectorlessRAGPipeline

vec = VectorRAGPipeline()
vl  = VectorlessRAGPipeline()

TEST_QUESTIONS = [
    "What was NVIDIA's total revenue for the most recent fiscal year?",
    "What percentage of NVIDIA's revenue came from data center products?",
    "What was Amazon Web Services revenue for the most recent fiscal year?",
    "What is Microsoft's stated strategy for artificial intelligence investments?",
    "What risks related to content licensing did Netflix identify?",
]

for q in TEST_QUESTIONS:
    print(f"\n{'#'*60}")
    print(f"  Q: {q}")
    print(f"{'#'*60}")

    print("Running Vector...")
    r_vec = vec.ask(q)

    print("Running Vectorless...")
    r_vl = vl.ask(q)

    print("Running Hybrid...")
    r_hybrid = hybrid.ask(q)

    print(f"\n[VECTOR]     {r_vec['answer'][:200]}")
    print(f"  ↳ time: {r_vec['total_time']}s")

    print(f"\n[BM25]       {r_vl['answer'][:200]}")
    print(f"  ↳ time: {r_vl['total_time']}s")

    print(f"\n[HYBRID]     {r_hybrid['answer'][:200]}")
    print(f"  ↳ time: {r_hybrid['total_time']}s  "
          f"(vector: {r_hybrid['vector_latency']}s  "
          f"bm25: {r_hybrid['bm25_latency']}s  "
          f"rerank: {r_hybrid['rerank_latency']}s)")
    print(f"  ↳ candidates: vector={r_hybrid['vector_candidates']} "
          f"bm25={r_hybrid['bm25_candidates']} "
          f"fused={r_hybrid['fused_candidates']} "
          f"final={len(r_hybrid['retrieved'])}")

🔧 Initialising Vector RAG Pipeline...
✅ ChromaDB loaded — 7913 child vectors
Mistral client already initialised - reusing
✅ Vector RAG ready — 2010 parents in lookup

🔧 Initialising Vectorless RAG Pipeline...
✅ BM25 loaded — 7913 children
Mistral client already initialised - reusing
✅ Vectorless RAG ready — 7913 children, 2010 parents


############################################################
  Q: What was NVIDIA's total revenue for the most recent fiscal year?
############################################################
Running Vector...
🔁 Loading reranker: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ Reranker ready
Running Vectorless...
Running Hybrid...

[VECTOR]     For **NVIDIA**, the total revenue for the most recent fiscal year (ended **January 25, 2026**) was **$215,938 million** (or **$215.94 billion**). This figure is sourced from the Consolidated Statement
  ↳ time: 11.1701s

[BM25]       For **NVIDIA**, the total revenue for the most recent fiscal year (2026) was **$215.9 billion**, reflecting a **65% increase** from the prior year. This figure is explicitly stated in the fiscal year 
  ↳ time: 1.2993s

[HYBRID]     For **NVIDIA**, the total revenue for the most recent fiscal year (ended **January 25, 2026**) was **$215,938 million** (or **$215.94 billion**). This figure is sourced from the Consolidated Statement
  ↳ time: 2.1115s  (vector: 0.3447s  bm25: 0.0404s  rerank: 0.2188s)
  ↳ candidates: vector=15 bm25=15 fused=15 final=5

############################################################
  Q: What percentage of NVIDIA's revenue came from data center products?
######

In [3]:
results = vec.collection.get(
    include=["metadatas"]
)

companies = set(
    m["company"]
    for m in results["metadatas"]
)

print(companies)
print(len(results["metadatas"]))

{'AMAZON', 'MICROSOFT', 'NETFLIX', 'NVIDIA'}
7913


In [4]:
q = "What risks related to content licensing did Netflix identify?"

ret = vec.ask(q)

print("Retrieved Chunks:", len(ret["retrieved"]))

for i, chunk in enumerate(ret["retrieved"]):
    print("\n" + "="*80)
    print(f"Chunk {i+1}")
    print("="*80)
    print(chunk["text"][:2000])

Retrieved Chunks: 5

Chunk 1
•
our ability to develop and expand an advertising sales and advertising technology organization team;
•
our ability to develop the technology, data, and related infrastructure to support advertising and drive value to advertisers;
•
the impact of our content and reputation on advertisers’ willingness to spend with us; and
•
any member dissatisfaction due to advertisements.
Risks Related to Intellectual Property
If studios, content providers or other rights holders refuse to license streaming content or other rights upon terms acceptable to us, our business could
be adversely affected.
Our ability to provide our members with content they can enjoy depends on obtaining various rights from third parties upon terms acceptable to us,
including necessary distribution rights, to such content and certain related elements thereof, such as the public performance of music contained within the

Chunk 2
compelling consumer proposition, piracy services are subject to ra

In [ ]:
from vector_rag.retriever     import VectorRAGPipeline

vec = VectorRAGPipeline()
vl  = VectorlessRAGPipeline()

ret = vec.retriever.retrieve(
    "What risks related to content licensing did Netflix identify?"
)

🔧 Initialising Vector RAG Pipeline...
✅ ChromaDB loaded — 7913 child vectors
Mistral client already initialised - reusing
✅ Vector RAG ready — 2010 parents in lookup

🔧 Initialising Vectorless RAG Pipeline...
✅ BM25 loaded — 7913 children
Mistral client already initialised - reusing
✅ Vectorless RAG ready — 7913 children, 2010 parents



AttributeError: 'VectorRAGPipeline' object has no attribute 'retriever'

In [ ]:
from vector_rag.retriever import retrieve

ret = retrieve(
    "What risks related to content licensing did Netflix identify?",
    vec.collection,
    vec.parent_lookup,
)

print(ret)

In [ ]:
from collections import Counter

results = vec.collection.get(include=["metadatas"])

print(Counter(
    m["company"]
    for m in results["metadatas"]
))


In [ ]:
import json

with open("../data/processed/chunks.json", encoding="utf-8") as f:
    data = json.load(f)

print("Parents:", len(data["parents"]))
print("Children:", len(data["children"]))

companies = {}

for c in data["children"]:
    company = c["company"]
    companies[company] = companies.get(company, 0) + 1

print(companies)

In [ ]:
import json

with open("../data/processed/chunks.json", encoding="utf-8") as f:
    data = json.load(f)

sources = {}

for c in data["children"]:
    company = c["company"]

    if company not in sources:
        sources[company] = set()

    sources[company].add(c["source"])

for company, s in sources.items():
    print(company)
    print(list(s)[:10])
    print()

In [ ]:
results = vec.collection.get(include=["metadatas"])

sources = {}

for m in results["metadatas"]:
    company = m["company"]

    if company not in sources:
        sources[company] = set()

    sources[company].add(m["source"])

for company, s in sources.items():
    print(company)
    print(list(s))

In [ ]:
import config
print(config.CHROMA_PERSIST_DIR)